# CLEIDS-Edge — Notebook 05: Post-Training Quantization & Pruning

Applies **post-training INT8 dynamic-range quantization** and **one-shot magnitude pruning** to all 10 already-trained CLEIDS-Edge checkpoints (5 datasets x binary/multiclass, from Notebook 03) — no retraining anywhere in this notebook, consistent with the project brief's "Edge adaptation" scope.

**CPU-only is fine for this notebook** — neither technique involves gradient computation; quantization is a one-time conversion, pruning is a direct weight-magnitude operation. A GPU runtime is not required (unlike Notebooks 03/04).

**Real technical fix required for TFLite conversion to work at all with this architecture** (found and verified locally before writing this notebook, not assumed): converting the LSTM-containing model to TFLite via the model's native dynamic-batch input signature fails outright — `TensorListReserve` requires a **static** batch size, so a dynamic (`None`) batch dimension errors during conversion. Rebuilding with a **fixed batch size baked into `Input()`** and `LSTM(..., unroll=True)` (same architecture, same weights via `set_weights`, different graph representation) is required. This export-only rebuild is used **exclusively for the TFLite artifact and quantization** — the original `.keras` checkpoints (used for every other number in this project) are never modified, and the "ORIGINAL" accuracy reported below is always from the true, unmodified checkpoint, not the export rebuild, so nothing here contaminates already-reported results.

**Evaluation protocol**: all compressed variants are evaluated at the **same default 0.5 threshold** (binary) as `main_results.json`, not the Youden's-J-tuned threshold from Notebook 03/04 — this notebook is scoped to "does compression preserve accuracy," not re-tuning the operating point. Multiclass uses argmax (no threshold concept). Every checkpoint's ORIGINAL (uncompressed) accuracy is printed and should match `main_results.json` closely — a real validation check that the loaded checkpoint and evaluation code are correct before trusting any compressed number.

**Techniques applied, per checkpoint**:
1. **Quantized**: post-training dynamic-range INT8 via `tf.lite.TFLiteConverter` (`Optimize.DEFAULT`) — weights become INT8, activations stay float, no calibration dataset needed.
2. **Pruned** at 30%, 50%, 70% sparsity: one-shot magnitude pruning (zero out the smallest-magnitude weights in each Dense/Conv1D/LSTM kernel) — genuinely zero fine-tuning, matching the project brief's "post-training... magnitude pruning" framing. Real, achievable size reduction measured via gzip compression of the saved model (standard practice in the pruning literature for reporting storage savings without specialized sparse-matrix runtime support).
3. **Combined**: 50%-pruned model, then quantized — the common real-world combination for maximum size reduction.

**No fabricated numbers** — every metric below comes from a real forward pass on this project's actual test data, same discipline as every other notebook.

**Incremental backup**: saves + Drive backup + GitHub push happen after **every single checkpoint**, not deferred to the end — Notebook 04 lost two full baselines' worth of real results to exactly that mistake (see `CLEIDS_PROJECT_BRIEF.md` §3b), not repeating it here.

## 1. Repo setup (clone/pull + auth)

In [ ]:
import os
import subprocess

REPO_URL = "https://github.com/NehlTech/CLEIDS-Edge.git"
REPO_DIR = "/content/CLEIDS-Edge"

GITHUB_TOKEN = None
try:
    from google.colab import userdata
    GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
except Exception as e:
    print(f"[DEBUG] userdata.get('GITHUB_TOKEN') raised {type(e).__name__}: {e}")
    GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN")

if not GITHUB_TOKEN:
    raise RuntimeError(
        "GITHUB_TOKEN not found. Add it as a Colab secret (key icon in the left sidebar) "
        "if running in the real Colab UI, or set os.environ['GITHUB_TOKEN'] manually for "
        "this session if running over a proxied connection."
    )

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

AUTH_REMOTE = REPO_URL.replace("https://", f"https://{GITHUB_TOKEN}@")
subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", AUTH_REMOTE], check=True)
subprocess.run(["git", "-C", REPO_DIR, "config", "user.email", "obololastkiller@gmail.com"])
subprocess.run(["git", "-C", REPO_DIR, "config", "user.name", "Bright Adu-Boahene"])

os.chdir(REPO_DIR)
print("Working directory:", os.getcwd())


## 2. Google Drive mount

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive/CLEIDS_Edge"
DRIVE_MODELS = os.path.join(DRIVE_ROOT, "models")
DRIVE_RESULTS = os.path.join(DRIVE_ROOT, "results")
DRIVE_DATA_PROCESSED = os.path.join(DRIVE_ROOT, "data_processed")
for d in (DRIVE_MODELS, DRIVE_RESULTS):
    os.makedirs(d, exist_ok=True)
print("Drive ready at:", DRIVE_ROOT)


## 3. Data bridge — copy processed splits from Google Drive

In [ ]:
import shutil

DATASETS = ["nsl-kdd", "cicids2017", "unsw-nb15", "ton-iot", "iot-23"]

if not os.path.isdir(DRIVE_DATA_PROCESSED):
    raise RuntimeError(
        f"{DRIVE_DATA_PROCESSED} not found. Upload data/processed/ to Google Drive at "
        f"MyDrive/CLEIDS_Edge/data_processed/ first (same data Notebooks 03/04 used)."
    )

for name in DATASETS:
    src_dir = os.path.join(DRIVE_DATA_PROCESSED, name)
    dst_dir = f"data/processed/{name}"
    os.makedirs(dst_dir, exist_ok=True)
    for fn in ["test.npz", "label_classes.json", "feature_names.json"]:
        src = os.path.join(src_dir, fn)
        dst = os.path.join(dst_dir, fn)
        if not os.path.exists(src):
            raise RuntimeError(f"{src} not found on Drive. Re-upload data/processed/{name}/ and re-run.")
        if os.path.exists(dst):
            print(f"[skip] {dst} already present")
            continue
        shutil.copy2(src, dst)
        print(f"[copied] {dst} ({os.path.getsize(dst)/1e6:.1f} MB)")

manifest_dst = "data/processed/preprocessing_manifest.json"
if not os.path.exists(manifest_dst):
    shutil.copy2(os.path.join(DRIVE_DATA_PROCESSED, "preprocessing_manifest.json"), manifest_dst)

print("\nAll processed test-split data copied from Drive (train/val not needed -- no retraining in this notebook).")


## 4. Setup (CPU-only is fine — no training in this notebook)

In [ ]:
import sys
import time
import json
import gzip
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers as klayers
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix,
)

tf.random.set_seed(42)
np.random.seed(42)
print("TensorFlow version:", tf.__version__)
print("GPU visible:", tf.config.list_physical_devices("GPU"), "(not required for this notebook)")

sys.path.insert(0, os.path.join(REPO_DIR, "src"))

for d in ["results"]:
    os.makedirs(d, exist_ok=True)

with open("data/processed/preprocessing_manifest.json") as f:
    prep_manifest = json.load(f)

with open("results/main_results.json") as f:
    main_results = json.load(f)
print("Loaded CLEIDS-Edge's own main_results.json (reference for the real-checkpoint validation check).")

NUM_CLASSES = {}
for name in DATASETS:
    with open(f"data/processed/{name}/label_classes.json") as f:
        NUM_CLASSES[name] = len(json.load(f)["classes"])
print("Num classes per dataset:", NUM_CLASSES)


## 5. Utilities

`build_export_model` rebuilds CLEIDS-Edge's exact architecture (matching `src/models.py::build_cleids_edge`) but with a fixed batch size baked into `Input()` and `LSTM(unroll=True)` — required for TFLite conversion to succeed at all (see the intro cell). Used only to produce the TFLite artifact; the real `.keras` checkpoint's own predictions (dynamic batch, no unroll) are what "ORIGINAL" always refers to below.

In [ ]:
def calculate_fpr(y_true, y_pred, binary=True):
    cm = confusion_matrix(y_true, y_pred)
    if binary:
        tn, fp, fn, tp = cm.ravel()
        return float(fp / (fp + tn)) if (fp + tn) > 0 else 0.0
    fprs = []
    for i in range(len(cm)):
        tp = cm[i, i]
        fp = cm[:, i].sum() - tp
        fn = cm[i, :].sum() - tp
        tn = cm.sum() - (tp + fp + fn)
        fprs.append(fp / (fp + tn) if (fp + tn) > 0 else 0.0)
    return float(np.mean(fprs))


def compute_metrics(y_true_cls, y_score, binary, num_classes, class_names):
    if binary:
        y_pred_cls = (y_score >= 0.5).astype(int)
        try:
            auc = float(roc_auc_score(y_true_cls, y_score))
        except ValueError as e:
            auc = None
            print(f"[WARN] AUC could not be computed: {e}")
        acc = float(accuracy_score(y_true_cls, y_pred_cls))
        prec = float(precision_score(y_true_cls, y_pred_cls, zero_division=0))
        rec = float(recall_score(y_true_cls, y_pred_cls, zero_division=0))
        f1 = float(f1_score(y_true_cls, y_pred_cls, zero_division=0))
        fpr = calculate_fpr(y_true_cls, y_pred_cls, binary=True)
    else:
        y_pred_cls = np.argmax(y_score, axis=1)
        # Same fix as Notebook 03: roc_auc_score(multi_class="ovr") silently returns
        # nan for classes with zero real test occurrences instead of raising --
        # compute per-class AUC explicitly, excluding absent classes.
        present = [i for i in range(num_classes) if (y_true_cls == i).any()]
        try:
            y_true_bin_cols = [(y_true_cls == i).astype(int) for i in present]
            per_class_auc = [roc_auc_score(y_true_bin_cols[j], y_score[:, i]) for j, i in enumerate(present)]
            auc = float(np.mean(per_class_auc))
        except ValueError as e:
            auc = None
            print(f"[WARN] Multiclass AUC could not be computed: {e}")
        acc = float(accuracy_score(y_true_cls, y_pred_cls))
        prec = float(precision_score(y_true_cls, y_pred_cls, average="macro", zero_division=0))
        rec = float(recall_score(y_true_cls, y_pred_cls, average="macro", zero_division=0))
        f1 = float(f1_score(y_true_cls, y_pred_cls, average="macro", zero_division=0))
        fpr = calculate_fpr(y_true_cls, y_pred_cls, binary=False)
    return {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1, "auc": auc, "fpr": fpr}


def build_export_model(input_dim, num_classes, binary, batch_size):
    """TFLite-export-only rebuild: fixed batch_shape + LSTM(unroll=True). Same
    architecture as build_cleids_edge, weights transferred via set_weights --
    never used for the "ORIGINAL" (real checkpoint) numbers, only for the
    quantized artifact."""
    inputs = tf.keras.Input(batch_shape=(batch_size, input_dim, 1), name="input")
    x = klayers.Conv1D(64, kernel_size=3, activation="relu", name="conv1d_block1")(inputs)
    x = klayers.BatchNormalization(name="bn1")(x)
    x = klayers.MaxPooling1D(pool_size=2, name="pool1")(x)
    x = klayers.Conv1D(128, kernel_size=3, activation="relu", name="conv1d_block2")(x)
    x = klayers.BatchNormalization(name="bn2")(x)
    x = klayers.MaxPooling1D(pool_size=2, name="pool2")(x)
    x = klayers.Dropout(0.3, name="dropout_conv")(x)
    x = klayers.LSTM(100, return_sequences=False, unroll=True, name="lstm")(x)
    x = klayers.Dropout(0.3, name="dropout_lstm")(x)
    x = klayers.Dense(64, activation="relu", name="dense_1")(x)
    if binary:
        outputs = klayers.Dense(1, activation="sigmoid", name="output")(x)
    else:
        outputs = klayers.Dense(num_classes, activation="softmax", name="output")(x)
    return tf.keras.Model(inputs=inputs, outputs=outputs, name="CLEIDS_Edge_export")


def quantize_to_tflite(orig_model, input_dim, num_classes, binary, tflite_path, batch_size=512):
    export_model = build_export_model(input_dim, num_classes, binary, batch_size)
    export_model.set_weights(orig_model.get_weights())
    converter = tf.lite.TFLiteConverter.from_keras_model(export_model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    tflite_bytes = converter.convert()
    with open(tflite_path, "wb") as f:
        f.write(tflite_bytes)
    return len(tflite_bytes)


def evaluate_tflite(tflite_path, X, batch_size=512):
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()
    interpreter.allocate_tensors()
    n = X.shape[0]
    n_padded = int(np.ceil(n / batch_size)) * batch_size
    X_padded = np.zeros((n_padded,) + X.shape[1:], dtype=np.float32)
    X_padded[:n] = X
    out_dim = output_details[0]["shape"][-1]
    scores = np.zeros((n_padded, out_dim), dtype=np.float32)
    for start in range(0, n_padded, batch_size):
        chunk = X_padded[start:start + batch_size]
        interpreter.set_tensor(input_details[0]["index"], chunk)
        interpreter.invoke()
        scores[start:start + batch_size] = interpreter.get_tensor(output_details[0]["index"])
    return scores[:n].ravel() if out_dim == 1 else scores[:n]


def prune_model_weights(m, sparsity):
    """One-shot magnitude pruning -- zero out the smallest-magnitude `sparsity`
    fraction of each Dense/Conv1D/LSTM kernel. No fine-tuning, genuinely
    post-training, matching the project brief's stated scope."""
    m2 = tf.keras.models.clone_model(m)
    m2.set_weights(m.get_weights())
    for layer in m2.layers:
        if isinstance(layer, (tf.keras.layers.Dense, tf.keras.layers.Conv1D)):
            w = layer.get_weights()
            if not w:
                continue
            kernel = w[0]
            thresh = np.percentile(np.abs(kernel), sparsity * 100)
            w[0] = kernel * (np.abs(kernel) >= thresh)
            layer.set_weights(w)
        elif isinstance(layer, tf.keras.layers.LSTM):
            w = layer.get_weights()
            for wi in [0, 1]:
                thresh = np.percentile(np.abs(w[wi]), sparsity * 100)
                w[wi] = w[wi] * (np.abs(w[wi]) >= thresh)
            layer.set_weights(w)
    return m2


def get_gzip_size(filepath):
    gz_path = filepath + ".gz"
    with open(filepath, "rb") as f_in, gzip.open(gz_path, "wb") as f_out:
        f_out.writelines(f_in)
    size = os.path.getsize(gz_path)
    os.remove(gz_path)
    return size


def save_compression_results(new_results):
    """Additive load-then-update, same discipline as Notebook 04's
    save_baseline_results -- called after EVERY checkpoint, not deferred."""
    path = "results/compression_results.json"
    existing = {}
    if os.path.exists(path):
        with open(path) as f:
            existing = json.load(f)
    for ckpt_name, data in new_results.items():
        existing[ckpt_name] = data
    with open(path, "w") as f:
        json.dump(existing, f, indent=2)
    print(f"Wrote {path} ({len(existing)} checkpoints total)")
    return existing


def backup_and_push(ckpt_name):
    shutil.copy2("results/compression_results.json", os.path.join(DRIVE_RESULTS, "compression_results.json"))
    subprocess.run(["git", "-C", REPO_DIR, "add", "-A", "results/"], check=False)
    commit_res = subprocess.run(
        ["git", "-C", REPO_DIR, "commit", "-m", f"Notebook 05: quantization+pruning for {ckpt_name}"],
        capture_output=True, text=True,
    )
    print(commit_res.stdout, commit_res.stderr)
    if commit_res.returncode == 0:
        subprocess.run(["git", "-C", REPO_DIR, "push", "origin", "HEAD"], check=True)
        print(f"Pushed: {ckpt_name}")
    else:
        print("Nothing new to commit (or commit failed) -- see output above.")


## 6. Main loop — quantize + prune all 10 CLEIDS-Edge checkpoints

For each of 5 datasets x {binary, multiclass}: load the real checkpoint, evaluate it as-is (validated against `main_results.json`), then quantize, prune at 30/50/70% sparsity, and combine (50% pruned + quantized). Saves, backs up to Drive, and pushes to GitHub after every single checkpoint.

In [ ]:
PRUNE_SPARSITIES = [0.3, 0.5, 0.7]
COMBINED_SPARSITY = 0.5

for dataset_name in DATASETS:
    data_dir = f"data/processed/{dataset_name}"
    test_data = np.load(f"{data_dir}/test.npz")
    X_test = test_data["X_cnn"]
    input_dim = X_test.shape[1]
    with open(f"{data_dir}/label_classes.json") as f:
        class_names = json.load(f)["classes"]
    num_classes = len(class_names)

    for task, binary in [("binary", True), ("multiclass", False)]:
        ckpt_name = f"cleids_edge_{dataset_name}_{task}"
        ckpt_path = f"models/{ckpt_name}.keras"
        print("\n" + "=" * 70)
        print(f"[COMPRESSING] {ckpt_name}")
        print("=" * 70)

        orig_model = tf.keras.models.load_model(ckpt_path)
        y_true_cls = test_data["y_bin"].astype(int) if binary else test_data["y_multi"].astype(int)

        # ORIGINAL -- real checkpoint, dynamic batch, no unroll. Validated against
        # main_results.json rather than trusted blindly.
        t0 = time.time()
        orig_score = orig_model.predict(X_test, batch_size=512, verbose=0)
        orig_score = orig_score.ravel() if binary else orig_score
        orig_metrics = compute_metrics(y_true_cls, orig_score, binary, num_classes, class_names)
        known_acc = main_results.get(dataset_name, {}).get(task, {}).get("accuracy")
        match_str = (f"(main_results.json: {known_acc:.4f}, diff={abs(orig_metrics['accuracy']-known_acc):.4f})"
                     if known_acc is not None else "(no main_results.json reference found)")
        print(f"[ORIGINAL] Acc={orig_metrics['accuracy']:.4f} F1={orig_metrics['f1']:.4f} "
              f"FPR={orig_metrics['fpr']:.4f} {match_str} (predict took {time.time()-t0:.1f}s)")
        orig_size_kb = os.path.getsize(ckpt_path) / 1e3

        result = {"original": {**orig_metrics, "size_kb": round(orig_size_kb, 2)}}

        # QUANTIZED
        tflite_path = f"models/{ckpt_name}_quant.tflite"
        quant_size_bytes = quantize_to_tflite(orig_model, input_dim, num_classes, binary, tflite_path)
        quant_score = evaluate_tflite(tflite_path, X_test)
        quant_metrics = compute_metrics(y_true_cls, quant_score, binary, num_classes, class_names)
        quant_size_kb = quant_size_bytes / 1e3
        print(f"[QUANTIZED] Acc={quant_metrics['accuracy']:.4f} F1={quant_metrics['f1']:.4f} "
              f"FPR={quant_metrics['fpr']:.4f} | size={quant_size_kb:.1f}KB "
              f"({100*(1-quant_size_kb/orig_size_kb):.1f}% smaller)")
        result["quantized"] = {**quant_metrics, "size_kb": round(quant_size_kb, 2)}
        os.remove(tflite_path)  # keep repo/Drive lean -- results captured in JSON, artifact not needed long-term

        # PRUNED at multiple sparsities
        result["pruned"] = {}
        for sparsity in PRUNE_SPARSITIES:
            pruned_model = prune_model_weights(orig_model, sparsity)
            pruned_score = pruned_model.predict(X_test, batch_size=512, verbose=0)
            pruned_score = pruned_score.ravel() if binary else pruned_score
            pruned_metrics = compute_metrics(y_true_cls, pruned_score, binary, num_classes, class_names)
            pruned_path = f"/tmp/pruned_{int(sparsity*100)}.keras"
            pruned_model.save(pruned_path)
            pruned_gz_kb = get_gzip_size(pruned_path) / 1e3
            orig_gz_kb = get_gzip_size(ckpt_path) / 1e3
            os.remove(pruned_path)
            print(f"[PRUNED {int(sparsity*100)}%] Acc={pruned_metrics['accuracy']:.4f} F1={pruned_metrics['f1']:.4f} "
                  f"FPR={pruned_metrics['fpr']:.4f} | gzip={pruned_gz_kb:.1f}KB vs original gzip {orig_gz_kb:.1f}KB "
                  f"({100*(1-pruned_gz_kb/orig_gz_kb):.1f}% smaller)")
            result["pruned"][f"{int(sparsity*100)}pct"] = {
                **pruned_metrics, "gzip_size_kb": round(pruned_gz_kb, 2), "gzip_original_size_kb": round(orig_gz_kb, 2),
            }
            if sparsity == COMBINED_SPARSITY:
                combined_orig_model = pruned_model  # reuse for the combined variant below

        # COMBINED: 50%-pruned + quantized
        combined_tflite_path = f"models/{ckpt_name}_pruned_quant.tflite"
        combined_size_bytes = quantize_to_tflite(combined_orig_model, input_dim, num_classes, binary, combined_tflite_path)
        combined_score = evaluate_tflite(combined_tflite_path, X_test)
        combined_metrics = compute_metrics(y_true_cls, combined_score, binary, num_classes, class_names)
        combined_size_kb = combined_size_bytes / 1e3
        print(f"[COMBINED {int(COMBINED_SPARSITY*100)}%-pruned+quantized] Acc={combined_metrics['accuracy']:.4f} "
              f"F1={combined_metrics['f1']:.4f} FPR={combined_metrics['fpr']:.4f} | size={combined_size_kb:.1f}KB "
              f"({100*(1-combined_size_kb/orig_size_kb):.1f}% smaller than original)")
        result["combined_pruned_quantized"] = {**combined_metrics, "size_kb": round(combined_size_kb, 2),
                                                 "prune_sparsity": COMBINED_SPARSITY}
        os.remove(combined_tflite_path)

        save_compression_results({ckpt_name: result})
        backup_and_push(ckpt_name)

print("\nAll 10 CLEIDS-Edge checkpoints compressed and evaluated.")


## 7. Consolidated Summary — Accuracy vs. Compression Tradeoff

In [ ]:
with open("results/compression_results.json") as f:
    compression_results = json.load(f)

print("\n" + "=" * 110)
print("CLEIDS-Edge -- Notebook 05 Headline Summary (accuracy delta vs. size reduction)")
print("=" * 110)
header = (f"{'Checkpoint':<28} | {'Orig Acc':<8} | {'Quant Acc':<9} | {'Quant Size%':<11} | "
          f"{'Prune50 Acc':<11} | {'Prune50 Size%':<13} | {'Combined Acc':<12} | {'Combined Size%':<14}")
print(header)
print("-" * len(header))
for ckpt_name, r in compression_results.items():
    orig_acc = r["original"]["accuracy"]
    orig_size = r["original"]["size_kb"]
    quant_acc = r["quantized"]["accuracy"]
    quant_size_pct = 100 * r["quantized"]["size_kb"] / orig_size
    p50 = r["pruned"]["50pct"]
    p50_size_pct = 100 * p50["gzip_size_kb"] / p50["gzip_original_size_kb"]
    comb = r["combined_pruned_quantized"]
    comb_size_pct = 100 * comb["size_kb"] / orig_size
    print(f"{ckpt_name:<28} | {orig_acc:<8.4f} | {quant_acc:<9.4f} | {quant_size_pct:<10.1f}% | "
          f"{p50['accuracy']:<11.4f} | {p50_size_pct:<12.1f}% | {comb['accuracy']:<12.4f} | {comb_size_pct:<13.1f}%")
print("=" * 110)
print("Full details (all sparsity levels, precision/recall/FPR/AUC) are in results/compression_results.json.")


## 8. Final backup + push

In [ ]:
shutil.copy2("results/compression_results.json", os.path.join(DRIVE_RESULTS, "compression_results.json"))
subprocess.run(["git", "-C", REPO_DIR, "add", "-A", "results/", "notebooks/05_Quantization_Pruning.ipynb"], check=False)
commit_res = subprocess.run(
    ["git", "-C", REPO_DIR, "commit", "-m", "Notebook 05: final quantization+pruning results (all 10 checkpoints)"],
    capture_output=True, text=True,
)
print(commit_res.stdout, commit_res.stderr)
if commit_res.returncode == 0:
    subprocess.run(["git", "-C", REPO_DIR, "push", "origin", "HEAD"], check=True)
    print("Pushed final results.")
else:
    print("Nothing new to commit (or commit failed) -- see output above.")
